# Datamine BACKTR Process Example

This notebook demonstrates how to configure and run the **`backtr`** process wrapper in `dmstudio`.

## Process Description

## BACKTR

The process back-transforms data from a normal distribution.

For details of the method refer to:

GSLIB. Geostatistical Software Library and User's Guide by Clayton V.Deutsch and Andre G.Journel, published by Oxford University Press, Second Edition, 1998. ISBN 0-19-510015-8.

There are some differences between Datamine and GSLIB conventions. However, unless otherwise stated, Datamine conventions are used. Where required these are converted to GSLIB conventions by the process. Please see the copyright notice at the bottom of this article.

The SGSIM process includes an option that carries out a normal transformation prior to the simulation. However if the option is not used then the simulated values will be normally distributed, and **BACKTR** will be required to back transform them..

### Input Files:

* **in1** (Table):
  Input data file. This must include the normally distributed field NORMVAL that is to be
  back transformed.
  Required=Yes

* **refdist** (Table):
  Input file containing the transformation lookup table. The file must include the field
  ORIGREF , containing the original data values, and the field NORMREF defining the
  corresponding normal score values.
  Required=Yes

### Output Files:

* **out** (Block Model):
  Output file containing the back transformed values. This contains the same data as the
  IN file, but with the added back transformed data field BACKVAL .
  Required=Yes

### Fields:

* **normval** (Numeric : IN):
  Field in the input data file IN defining the normal score values to be back
  transformed.
  Default=Undefined
  Required=Yes

* **origref** (Numeric : REFDIST):
  Field in the input REFDIST file defining the original data values.
  Default=Undefined
  Required=Yes

* **normref** (Numeric : REFDIST):
  Field in the input REFDIST file defining the normal score values.
  Default=Undefined
  Required=Yes

* **backval** (Numeric : Undefined):
  Field to be created in the output file OUT for the back transformed data values. If not
  specified then the field BACKVAL will be created,
  Default=BACKVAL
  Required=No

### Parameters:

* **minnorm**:
  Minimum value of NORMVAL field in IN file to be used for back transformation.
  Range=Undefined
  Values=Undefined
  Default=-4
  Required=No

* **maxnorm**:
  Maximum value of NORMVAL field in IN file to be used for back transformation.
  Range=Undefined
  Values=Undefined
  Default=4
  Required=No

* **minback**:
  Minimum value of BACKVAL field to be created in OUT file.
  Range=Undefined
  Values=Undefined
  Default=0
  Required=No

* **maxback**:
  Maximum value of BACKVAL field to be created in OUT file.
  Range=Undefined
  Values=Undefined
  Default=+
  Required=No

* **lotail**:
  Back-transformation method in the lower tail of the distribution to a minimum grade of
  MINBACK . Options: **1,**: Linear interpolation.; **2,**: Power model interpolation. The
  power used is defined by LOPAR .
  Range=1,2
  Values=1,2
  Default=1
  Required=No

* **lopar**:
  Power parameter used in back-transformation of grades in the lower tail of the
  distribution to a minimum of MINBACK . LOTAIL must be set to 2.
  Range=0,+
  Values=Undefined
  Default=1
  Required=No

* **uptail**:
  Back-transformation method in the upper tail of the distribution to a maximum grade of
  MAXGRADE . Options: **1,**: Linear interpolation.; **2,**: Power model interpolation.
  The power used is defined by UPPAR .; **4,**: Hyperbolic model extrapolation using power
  parameter defined by UPPAR .
  Range=1,4
  Values=1,2,4
  Default=1
  Required=No

* **uppar**:
  Power parameter used in back-transformation of grades in the upper tail of the
  distribution to a maximum of MAXGRADE . UPTAIL must be set to 2 or 4.
  Range=0,+
  Values=Undefined
  Default=1
  Required=No

In [ ]:
# Step 1: Connect to Datamine and Initialize Sandbox
import os
import shutil
import glob

import pandas as pd

from dmstudio import dmcommands, dmfiles, initialize, agent

# Connect to running Studio RM instance
dm_cmd = dmcommands.init(version='StudioRM')
dm_fil = dmfiles.init(version='StudioRM')
oScript = initialize.studio('StudioRM')
print(f"Connected to: {oScript.Caption}")

# Initialize active project sandbox using the shared test_sandbox project
notebook_folder = os.path.normpath(os.path.dirname(os.path.abspath('__file__'))).lower()
agent.initialize_sandbox(notebook_folder)

## Step 2: Introspect Schema
Always inspect the parameter schema for the process wrapper to see the expected input and output files, fields, and parameters.

In [ ]:
schema = agent.get_command_schema('backtr')
print(f"Process: {schema['name']}")
print("Parameters:")
for p in schema['parameters']:
    print(f"  - {p['name']}: default={p['default']}, annotation={p['annotation']}")

## Step 3: Prepare Inputs
We initialize the example project by copying the relevant standard datasets from the Datamine database locally to this sandbox folder using a `t_` prefix.

In [ ]:
# Step 3: Copy VBOP datasets dynamically from Database to test_sandbox
files_to_copy = [
    "_vb_assays.dmx",
    "_vb_collars.dmx",
    "_vb_surveys.dmx",
    "_vb_lithology.dmx",
    "_vb_epar.dmx",
    "_vb_spar.dmx",
    "_vb_vpar.dmx",
    "_vb_mod1.dmx",
    "_vb_SurfacePointsPt.dmx",
    "_vb_SurfaceTriangles.dmx"
]

agent.copy_database_files(files_to_copy)

## Step 4: Execute Process
Call the wrapper method with appropriate arguments. Below is the generated template execution call. Required parameters are active, and optional parameters are commented out.

In [ ]:
# Execute backtr
print("Running backtr...")
dm_cmd.backtr(
    in1_i='t_input_file',  # required
    refdist_i='t_input_file',  # required
    out_o='t_backtr_out',  # required
    normval_f='optional',  # required
    origref_f='optional',  # required
    normref_f='optional',  # required
    # backval_f='optional',  # optional
    # minnorm_p=-4,  # optional
    # maxnorm_p=4,  # optional
    # minback_p=0,  # optional
    # maxback_p='+',  # optional
    # lotail_p=1,  # optional
    # lopar_p=1,  # optional
    # uptail_p=1,  # optional
    # uppar_p=1,  # optional
    # arguments='optional',  # optional
    # retrieval='optional',  # optional
)
print("backtr execution completed.")

## Step 5: Verify Results
Check that output files exist on disk and read them using pandas to verify the outputs.

In [ ]:
# Step 5: Verify results
output_file = 't_backtr_out.dmx'
if os.path.exists(output_file):
    df = agent.read_datamine(output_file)
    print(f"Output file loaded successfully. Rows: {len(df)}")
    print(df.head())
else:
    print("Output file not found (run Step 4 first)")

## Step 6: Clean up Project Folder
Always clean up generated temporary files to keep the directory clean.

In [ ]:
# Step 6: Clean up temporary files and generated artifacts
# 1. Clean up temporary files matching t_*.*
temp_files = glob.glob("t_*.*")
for f in temp_files:
    try:
        os.remove(f)
        print(f"Removed {os.path.basename(f)}")
    except Exception as e:
        print(f"Failed to remove {os.path.basename(f)}: {e}")

# 2. Clean up dynamic python initialization files (dmdir.py, __init__.py)
extra_files = ['dmdir.py', '__init__.py']
for f in extra_files:
    if os.path.exists(f):
        try:
            os.remove(f)
            print(f"Removed {f}")
        except Exception as e:
            print(f"Failed to remove {f}: {e}")

# 3. Clean up cache directories (__pycache__)
pycache = '__pycache__'
if os.path.exists(pycache):
    try:
        shutil.rmtree(pycache)
        print("Removed __pycache__")
    except Exception as e:
        print(f"Failed to remove __pycache__: {e}")